# Detección de drift: PSI, KS y por qué una alerta no es un diagnóstico

**Perfil:** Liderazgo de datos y gobierno  
**Datos:** sintéticos con semilla fija; se conoce el proceso generador, así que se puede comprobar si el método recupera la verdad.

## Pregunta que guía el notebook

Un tablero de monitoreo marca variables en ámbar y rojo. ¿Cuánto se puede confiar? El PSI y el KS tienen **ruido de muestreo**, **umbrales** que dependen del tamaño de muestra, **puntos ciegos** (cambios que no ven) y, con muchas variables, **falsas alarmas garantizadas**. Además, que una variable derive **no implica** que el modelo empeore, y que el modelo empeore **no implica** que haya drift visible.

## Recorrido

1. **PSI**: definición, valor poblacional exacto bajo un desplazamiento normal y verificación por simulación.
2. **Ruido del PSI sin drift** y por qué el umbral 0,1 falla con muestras chicas.
3. **KS y tamaño de muestra:** significancia estadística frente a relevancia práctica.
4. **Poder de detección** de PSI, KS, prueba de media, de varianza y de Wasserstein ante cuatro tipos de cambio, a igual tasa de falsas alarmas.
5. **Variables categóricas:** categorías nuevas o desaparecidas y la sensibilidad al $\varepsilon$.
6. **Drift multivariado:** cambio de correlación invisible a las pruebas univariadas; clasificador de dos muestras.
7. **Múltiples variables:** falsas alarmas esperadas y corrección (Bonferroni, Benjamini–Hochberg).
8. **Drift y desempeño del modelo:** tres casos que muestran que no van de la mano.
9. Pruebas y límites.

Las conclusiones numéricas se **calculan y se imprimen desde los datos** (celdas «Lectura»), y las pruebas de la sección final hacen fallar el notebook si un hallazgo deja de cumplirse.

## 1. Configuración

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
SEED = 2026
plt.rcParams.update({"figure.figsize": (7.5, 4), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.options.display.float_format = "{:,.4f}".format
from scipy import stats
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss

## 2. PSI: definición y valor esperado

Con $k$ cajas definidas por los **cuantiles de la muestra de referencia**, si $e_i$ y $a_i$ son las proporciones de referencia y actual en la caja $i$:
$$\text{PSI}=\sum_{i=1}^k (a_i-e_i)\ln\frac{a_i}{e_i}.$$
Es la **divergencia de Jeffreys** discretizada. Para una variable $N(0,1)$ que se desplaza a $N(\delta,1)$, con cajas poblacionales de cuantiles de $N(0,1)$, el PSI poblacional se calcula **exactamente** con la CDF normal: $e_i=1/k$ y $a_i=\Phi(q_{i+1}-\delta)-\Phi(q_i-\delta)$. Se compara con la estimación muestral (n=400.000).

In [2]:
EPS = 1e-6
def psi(ref, act, k=10, edges=None, eps=EPS):
    if edges is None: edges = np.quantile(ref, np.linspace(0, 1, k + 1)[1:-1])
    e = np.bincount(np.searchsorted(edges, ref, side="right"), minlength=len(edges) + 1) / len(ref)
    a = np.bincount(np.searchsorted(edges, act, side="right"), minlength=len(edges) + 1) / len(act)
    e, a = np.clip(e, eps, None), np.clip(a, eps, None); return float(((a - e) * np.log(a / e)).sum())

def psi_poblacional(delta, sigma=1.0, k=10):
    q = stats.norm.ppf(np.linspace(0, 1, k + 1)); cdf = stats.norm.cdf((q - delta) / sigma); a = np.diff(cdf); e = np.full(k, 1 / k); return float(((a - e) * np.log(a / e)).sum())

rg = np.random.default_rng(SEED); N = 400_000; ref = rg.standard_normal(N)
filas = []
for d in (0.0, 0.1, 0.25, 0.5, 1.0):
    cur = rg.standard_normal(N) + d; filas.append({"desplazamiento_delta": d, "PSI_poblacional": psi_poblacional(d), "PSI_muestral": psi(ref, cur), "J_continua = delta^2": d ** 2})
pp = pd.DataFrame(filas).set_index("desplazamiento_delta"); pp.round(4)

,PSI_poblacional,PSI_muestral,J_continua = delta^2
desplazamiento_delta,,,
0.0000,0.0000,0.0000,0.0000
0.1000,0.0096,0.0093,0.0100
0.2500,0.0598,0.0594,0.0625
0.5000,0.2377,0.2389,0.2500
1.0000,0.9261,0.9257,1.0000


In [3]:
print("Lectura:")
print(f"  La estimacion muestral coincide con el valor poblacional exacto (error maximo {np.abs(pp.PSI_muestral - pp.PSI_poblacional).max():.4f}): la implementacion es correcta.")
print(f"  El PSI con 10 cajas es MENOR que la divergencia continua (delta^2): para delta=1, {pp.loc[1.0, 'PSI_poblacional']:.2f} frente a 1.00. Discretizar pierde informacion; mas cajas se acercan al limite.")
print(f"  La regla 'PSI > 0.1 = ambar' corresponde, con 10 cajas, a un desplazamiento de ~{np.interp(0.1, pp.PSI_poblacional, pp.index):.2f} desviaciones estandar; 'PSI > 0.25 = rojo' a ~{np.interp(0.25, pp.PSI_poblacional, pp.index):.2f}.")

Lectura:
  La estimacion muestral coincide con el valor poblacional exacto (error maximo 0.0012): la implementacion es correcta.
  El PSI con 10 cajas es MENOR que la divergencia continua (delta^2): para delta=1, 0.93 frente a 1.00. Discretizar pierde informacion; mas cajas se acercan al limite.
  La regla 'PSI > 0.1 = ambar' corresponde, con 10 cajas, a un desplazamiento de ~0.31 desviaciones estandar; 'PSI > 0.25 = rojo' a ~0.51.


## 3. El PSI también tiene ruido sin drift

Si las dos muestras vienen de la **misma** distribución, el PSI no es 0: $\text{PSI}\approx\chi^2_{k-1}\,(1/n+1/m)/\,$(...), con media aproximada $(k-1)\left(\frac1n+\frac1m\right)$. Con $n=m=200$ y $k=10$ eso es $0{,}09$: casi el umbral de alerta *sin ningún cambio*. Se mide la distribución del PSI bajo la nula y la tasa de «alertas» con los umbrales 0,1 y 0,25.

In [4]:
filas = []
for n in (100, 200, 500, 2000, 10_000):
    rg = np.random.default_rng(n); vals = np.array([psi(rg.standard_normal(n), rg.standard_normal(n)) for _ in range(1500)])
    filas.append({"n_por_muestra": n, "PSI_medio": vals.mean(), "aprox (k-1)(2/n)": 9 * 2 / n, "P95": np.percentile(vals, 95), "alerta_ambar(>0.1)": (vals > 0.1).mean(), "alerta_roja(>0.25)": (vals > 0.25).mean()})
nula = pd.DataFrame(filas).set_index("n_por_muestra"); nula.round(4)

,PSI_medio,aprox (k-1)(2/n),P95,alerta_ambar(>0.1),alerta_roja(>0.25)
n_por_muestra,,,,,
100,0.2057,0.1800,0.3910,0.8540,0.2473
200,0.0932,0.0900,0.1769,0.3773,0.0027
500,0.0361,0.0360,0.0673,0.0007,0.0000
2000,0.0089,0.0090,0.0167,0.0000,0.0000
10000,0.0018,0.0018,0.0033,0.0000,0.0000


In [5]:
print("Lectura:")
print(f"  El PSI medio sin drift sigue (k-1)(1/n+1/m): {nula.loc[200, 'PSI_medio']:.3f} con n=200 (aprox. {nula.loc[200, 'aprox (k-1)(2/n)']:.3f}).")
print(f"  Con n=100 por muestra, {nula.loc[100, 'alerta_ambar(>0.1)']:.0%} de los lotes SIN cambio alguno quedan en ambar y {nula.loc[100, 'alerta_roja(>0.25)']:.1%} en rojo; con n=2.000 la tasa de ambar baja a {nula.loc[2000, 'alerta_ambar(>0.1)']:.1%}.")
print(f"  Un umbral fijo ignora el tamano de muestra: lo correcto es calibrarlo con la distribucion nula (p. ej. P95 = {nula.loc[200, 'P95']:.3f} con n=200, {nula.loc[2000, 'P95']:.4f} con n=2.000).")

Lectura:
  El PSI medio sin drift sigue (k-1)(1/n+1/m): 0.093 con n=200 (aprox. 0.090).
  Con n=100 por muestra, 85% de los lotes SIN cambio alguno quedan en ambar y 24.7% en rojo; con n=2.000 la tasa de ambar baja a 0.0%.
  Un umbral fijo ignora el tamano de muestra: lo correcto es calibrarlo con la distribucion nula (p. ej. P95 = 0.177 con n=200, 0.0167 con n=2.000).


## 4. KS: significancia frente a relevancia

La prueba de Kolmogorov–Smirnov compara las funciones de distribución. Con muestras enormes **cualquier** diferencia, por irrelevante que sea, resulta «significativa». Se mide, para un desplazamiento diminuto de $\delta=0{,}03$ desviaciones, la tasa de rechazo del KS ($p<0{,}05$) y el PSI mediano según el tamaño de muestra.

In [6]:
filas = []
for n in (500, 5_000, 50_000, 500_000):
    R = 200 if n <= 50_000 else 60; rg = np.random.default_rng(n); rech = []; ps = []
    for _ in range(R):
        a = rg.standard_normal(n); b = rg.standard_normal(n) + 0.03; rech.append(stats.ks_2samp(a, b).pvalue < 0.05); ps.append(psi(a, b))
    filas.append({"n_por_muestra": n, "rechazo_KS(p<.05)": np.mean(rech), "PSI_mediano": np.median(ps), "PSI_poblacional": psi_poblacional(0.03)})
ks = pd.DataFrame(filas).set_index("n_por_muestra"); ks.round(5)

,rechazo_KS(p<.05),PSI_mediano,PSI_poblacional
n_por_muestra,,,
500,0.0850,0.0360,0.0009
5000,0.2050,0.0043,0.0009
50000,0.9900,0.0012,0.0009
500000,1.0000,0.0009,0.0009


In [7]:
print("Lectura (desplazamiento real de 0.03 desviaciones, practicamente irrelevante):")
print(f"  El KS rechaza en {ks.loc[500, 'rechazo_KS(p<.05)']:.0%} de los casos con n=500 y en {ks.loc[500_000, 'rechazo_KS(p<.05)']:.0%} con n=500.000: la 'alerta' crece con n aunque el cambio sea el mismo.")
print(f"  El PSI poblacional de ese cambio es {ks.loc[500, 'PSI_poblacional']:.5f}, 100 veces bajo el umbral ambar: el PSI mide magnitud; el KS mide evidencia. Un buen monitoreo reporta ambos y alerta con una regla de magnitud + significancia.")

Lectura (desplazamiento real de 0.03 desviaciones, practicamente irrelevante):
  El KS rechaza en 8% de los casos con n=500 y en 100% con n=500.000: la 'alerta' crece con n aunque el cambio sea el mismo.
  El PSI poblacional de ese cambio es 0.00086, 100 veces bajo el umbral ambar: el PSI mide magnitud; el KS mide evidencia. Un buen monitoreo reporta ambos y alerta con una regla de magnitud + significancia.


## 5. Poder de detección según el tipo de cambio

Cuatro cambios (n=2.000 por muestra), todos «reales»: **desplazamiento de media** (0,15), **aumento de varianza** (×1,3 en desviación), **mezcla bimodal** con la *misma media y varianza* que la referencia ($0{,}5\,N(-0{,}8,0{,}6^2)+0{,}5\,N(0{,}8,0{,}6^2)$) y **colas pesadas** (t de Student con 3 g.l. reescalada a varianza 1). Estadísticos: PSI, KS, \|t\| de medias, \|log de la razón de varianzas\| (sensible solo a la dispersión), Levene (dispersión alrededor de la mediana, que también reacciona a la forma) y distancia de Wasserstein. Para comparar de forma justa, el valor crítico de cada estadístico se calibra para una **tasa de falsas alarmas del 5 %** con simulación bajo la nula.

In [8]:
n = 2000
def stat_set(a, b):
    return {"PSI": psi(a, b), "KS": stats.ks_2samp(a, b).statistic, "|t| medias": abs(stats.ttest_ind(a, b, equal_var=False).statistic),
            "razon var": abs(np.log(a.var(ddof=1) / b.var(ddof=1))), "Levene": stats.levene(a, b).statistic, "Wasserstein": stats.wasserstein_distance(a, b)}
def gen_ref(rg): return rg.standard_normal(n)
CAMBIOS = {"media +0.15": lambda rg: rg.standard_normal(n) + 0.15,
           "desvio x1.3": lambda rg: 1.3 * rg.standard_normal(n),
           "bimodal (misma media/var)": lambda rg: np.where(rg.random(n) < .5, -0.8, 0.8) + 0.6 * rg.standard_normal(n),
           "colas pesadas t3 (var 1)": lambda rg: rg.standard_t(3, n) / np.sqrt(3.0)}
rg = np.random.default_rng(77); R = 400
nulo_ = pd.DataFrame([stat_set(gen_ref(rg), gen_ref(rg)) for _ in range(R)]); crit = nulo_.quantile(0.95)
filas = []
for nombre, gen in CAMBIOS.items():
    alt = pd.DataFrame([stat_set(gen_ref(rg), gen(rg)) for _ in range(R)]); filas.append({"cambio": nombre, **{c: (alt[c] > crit[c]).mean() for c in crit.index}})
poder = pd.DataFrame(filas).set_index("cambio"); print("Valor critico (P95 de la nula):", crit.round(4).to_dict()); poder.round(2)

Valor critico (P95 de la nula): {'PSI': 0.016, 'KS': 0.042, '|t| medias': 1.9097, 'razon var': 0.0902, 'Levene': 4.1087, 'Wasserstein': 0.0654}


,PSI,KS,|t| medias,razon var,Levene,Wasserstein
cambio,,,,,,
media +0.15,0.9500,0.9900,1.0000,0.0500,0.0500,1.0000
desvio x1.3,1.0000,1.0000,0.0600,1.0000,1.0000,1.0000
bimodal (misma media/var),1.0000,1.0000,0.0400,0.0200,0.8100,1.0000
colas pesadas t3 (var 1),1.0000,1.0000,0.0500,0.5600,1.0000,1.0000


In [9]:
print("Lectura (poder = fraccion de lotes con cambio real detectados, a 5% de falsas alarmas):")
for c in poder.index:
    print(f"  {c:28s}: " + ", ".join(f"{k} {v:.0%}" for k, v in poder.loc[c].items()))
bi = poder.loc["bimodal (misma media/var)"]
print(f"  La mezcla bimodal con igual media y varianza es INVISIBLE para la prueba de medias ({bi['|t| medias']:.0%}) y para la razon de varianzas ({bi['razon var']:.0%}), es decir, al nivel de falsas alarmas (5%); el KS ({bi['KS']:.0%}) y el PSI ({bi['PSI']:.0%}) la ven.")
print(f"  Levene la detecta en {bi['Levene']:.0%}: no es una prueba pura de varianza, reacciona a la forma (la distribucion bimodal tiene desviaciones absolutas respecto de la mediana distintas). Ningun estadistico unico cubre todos los cambios.")

Lectura (poder = fraccion de lotes con cambio real detectados, a 5% de falsas alarmas):
  media +0.15                 : PSI 95%, KS 99%, |t| medias 100%, razon var 5%, Levene 5%, Wasserstein 100%
  desvio x1.3                 : PSI 100%, KS 100%, |t| medias 6%, razon var 100%, Levene 100%, Wasserstein 100%
  bimodal (misma media/var)   : PSI 100%, KS 100%, |t| medias 4%, razon var 2%, Levene 81%, Wasserstein 100%
  colas pesadas t3 (var 1)    : PSI 100%, KS 100%, |t| medias 5%, razon var 56%, Levene 100%, Wasserstein 100%
  La mezcla bimodal con igual media y varianza es INVISIBLE para la prueba de medias (4%) y para la razon de varianzas (2%), es decir, al nivel de falsas alarmas (5%); el KS (100%) y el PSI (100%) la ven.
  Levene la detecta en 81%: no es una prueba pura de varianza, reacciona a la forma (la distribucion bimodal tiene desviaciones absolutas respecto de la mediana distintas). Ningun estadistico unico cubre todos los cambios.


## 6. Variables categóricas

Para una variable categórica el PSI es la misma suma sobre categorías. El problema práctico: una categoría **nueva** (proporción de referencia 0) o **desaparecida** (proporción actual 0) hace explotar el logaritmo. El $\varepsilon$ con el que se reemplazan los ceros **decide** el resultado.

In [10]:
ref_c = pd.Series({"A": .50, "B": .30, "C": .20, "D": 0.0}); act_c = pd.Series({"A": .45, "B": .28, "C": .17, "D": .10})        # aparece la categoria D (10%)
des_c = pd.Series({"A": .62, "B": .38, "C": 0.0, "D": 0.0}); ref2 = pd.Series({"A": .50, "B": .30, "C": .20, "D": 0.0})          # desaparece C
def psi_cat(e, a, eps):
    e, a = np.clip(e.values, eps, None), np.clip(a.values, eps, None); return float(((a - e) * np.log(a / e)).sum())
filas = [{"epsilon": eps, "PSI categoria nueva (10%)": psi_cat(ref_c, act_c, eps), "PSI categoria desaparecida (20%)": psi_cat(ref2, des_c, eps)} for eps in (1e-2, 1e-3, 1e-4, 1e-6, 1e-9)]
cat = pd.DataFrame(filas).set_index("epsilon"); cat.round(3)

,PSI categoria nueva (10%),PSI categoria desaparecida (20%)
epsilon,,
0.0100,0.2190,0.6140
0.0010,0.4670,1.0990
0.0001,0.7020,1.5640
0.0000,1.1630,2.4860
0.0000,1.8540,3.8670


In [11]:
print("Lectura:")
print(f"  El PSI de una categoria nueva con 10% de peso va de {cat.iloc[0, 0]:.2f} (eps=1e-2) a {cat.iloc[-1, 0]:.2f} (eps=1e-9): crece como ln(1/eps).")
print(f"  El resultado cambia en un factor {cat.iloc[-1, 0] / cat.iloc[0, 0]:.1f} solo por elegir eps: la cifra es una convencion, no una medida. Es mejor reportar APARTE 'categorias nuevas/desaparecidas y su peso' (informacion exacta) y usar el PSI como magnitud solo del resto.")

Lectura:
  El PSI de una categoria nueva con 10% de peso va de 0.22 (eps=1e-2) a 1.85 (eps=1e-9): crece como ln(1/eps).
  El resultado cambia en un factor 8.5 solo por elegir eps: la cifra es una convencion, no una medida. Es mejor reportar APARTE 'categorias nuevas/desaparecidas y su peso' (informacion exacta) y usar el PSI como magnitud solo del resto.


## 7. Drift multivariado: la correlación cambia, los marginales no

Referencia: $(X_1,X_2)\sim N(0,\Sigma)$ con correlación $+0{,}8$. Actual: correlación $-0{,}8$. Los **marginales** son idénticos ($N(0,1)$), así que el KS por variable no ve nada. Un **clasificador de dos muestras** (¿puedo distinguir si una fila es de referencia o actual?) sí: un AUC de validación cruzada muy superior a 0,5 indica drift. La nula del AUC se calibra permutando etiquetas.

In [12]:
def muestra(rho, n, rg): 
    z = rg.standard_normal((n, 2)); return np.c_[z[:, 0], rho * z[:, 0] + np.sqrt(1 - rho ** 2) * z[:, 1]]

def auc_dos_muestras(A, B, seed=0, etiquetas=None):
    X = np.r_[A, B]; y = np.r_[np.zeros(len(A)), np.ones(len(B))] if etiquetas is None else etiquetas
    p = cross_val_predict(HistGradientBoostingClassifier(max_iter=80, random_state=0), X, y, cv=StratifiedKFold(5, shuffle=True, random_state=seed), method="predict_proba")[:, 1]
    return roc_auc_score(y, p)

rg = np.random.default_rng(12); n = 1500
A = muestra(0.8, n, rg); B = muestra(-0.8, n, rg); A2 = muestra(0.8, n, rg)
ks_vars = [stats.ks_2samp(A[:, j], B[:, j]).pvalue for j in range(2)]
auc_real = auc_dos_muestras(A, B); auc_sin = auc_dos_muestras(A, A2)
X = np.r_[A, B]; y = np.r_[np.zeros(n), np.ones(n)]; nulo_auc = [auc_dos_muestras(A, B, etiquetas=rg.permutation(y)) for _ in range(30)]
ks_vars = [float(p) for p in ks_vars]
print(f"p-valores KS por variable: {[round(p, 3) for p in ks_vars]}  (marginales identicos)")
print(f"AUC del clasificador: correlacion cambiada = {auc_real:.3f};  sin cambio (dos muestras de la referencia) = {auc_sin:.3f};  nula por permutacion: media {np.mean(nulo_auc):.3f}, max {np.max(nulo_auc):.3f}")

p-valores KS por variable: [0.539, 0.752]  (marginales identicos)
AUC del clasificador: correlacion cambiada = 0.862;  sin cambio (dos muestras de la referencia) = 0.502;  nula por permutacion: media 0.499, max 0.518


In [13]:
print("Lectura:")
print(f"  Las pruebas univariadas no detectan nada (p = {ks_vars[0]:.2f} y {ks_vars[1]:.2f}); el clasificador separa las muestras con AUC {auc_real:.2f}, muy por encima de la nula (max {np.max(nulo_auc):.2f}).")
print("  Con dos muestras de la MISMA distribucion el AUC queda en ~0.5: el metodo no inventa drift.")
print("  Ademas el clasificador permite diagnosticar: la importancia de las variables / interacciones indica donde esta el cambio.")

Lectura:
  Las pruebas univariadas no detectan nada (p = 0.54 y 0.75); el clasificador separa las muestras con AUC 0.86, muy por encima de la nula (max 0.52).
  Con dos muestras de la MISMA distribucion el AUC queda en ~0.5: el metodo no inventa drift.
  Ademas el clasificador permite diagnosticar: la importancia de las variables / interacciones indica donde esta el cambio.


## 8. Muchas variables: falsas alarmas esperadas

Con $m$ variables y un KS al 5 % por variable, aun sin ningún cambio la probabilidad de **al menos una alarma** por lote es $1-0{,}95^m$. Para $m=50$: 92 %. Se compara el KS crudo con **Bonferroni** ($\alpha/m$) y **Benjamini–Hochberg** (FDR 5 %) en un escenario con 50 variables, de las cuales **3** derivan ($\delta=0{,}25$, $n=1.000$), a lo largo de 300 lotes.

In [14]:
m, n_, LOTES, DER = 50, 1000, 300, 3
rg = np.random.default_rng(21); res = {"sin_drift": {"crudo": [], "bonf": [], "bh": []}, "con_drift": {"crudo": [], "bonf": [], "bh": []}}
def bh(p, q=0.05):
    o = np.argsort(p); k = np.nonzero(p[o] <= q * (np.arange(1, len(p) + 1) / len(p)))[0]; sel = np.zeros(len(p), bool)
    if len(k): sel[o[:k.max() + 1]] = True
    return sel
tp = {"crudo": 0, "bonf": 0, "bh": 0}; fp = {"crudo": 0, "bonf": 0, "bh": 0}; sel_tot = {"crudo": 0, "bonf": 0, "bh": 0}
for l in range(LOTES):
    p0 = np.array([stats.ks_2samp(rg.standard_normal(n_), rg.standard_normal(n_)).pvalue for _ in range(m)])
    sel0 = {"crudo": p0 < .05, "bonf": p0 < .05 / m, "bh": bh(p0)}
    for k_, s_ in sel0.items(): res["sin_drift"][k_].append(s_.any())
    p1 = np.array([stats.ks_2samp(rg.standard_normal(n_), rg.standard_normal(n_) + (0.25 if j < DER else 0)).pvalue for j in range(m)])
    sel1 = {"crudo": p1 < .05, "bonf": p1 < .05 / m, "bh": bh(p1)}
    for k_, s_ in sel1.items(): tp[k_] += s_[:DER].sum(); fp[k_] += s_[DER:].sum(); sel_tot[k_] += s_.sum(); res["con_drift"][k_].append(s_.any())
mt = pd.DataFrame({"P(alguna alarma | sin drift)": {k_: np.mean(v) for k_, v in res["sin_drift"].items()},
                   "poder por variable con drift": {k_: tp[k_] / (LOTES * DER) for k_ in tp}, "falsas alarmas por lote (con drift)": {k_: fp[k_] / LOTES for k_ in fp},
                   "FDR observado": {k_: fp[k_] / max(sel_tot[k_], 1) for k_ in fp}}); mt.round(3)

,P(alguna alarma | sin drift),poder por variable con drift,falsas alarmas por lote (con drift),FDR observado
crudo,0.9200,0.9990,2.3230,0.4370
bonf,0.0500,0.9080,0.0400,0.0140
bh,0.0500,0.9580,0.1830,0.0600


In [15]:
print(f"Lectura (teoria: 1 - 0.95^{m} = {1 - 0.95 ** m:.1%}):")
print(f"  Sin ningun drift, el KS crudo da al menos una alarma en {mt.loc['crudo', 'P(alguna alarma | sin drift)']:.0%} de los lotes; Bonferroni {mt.loc['bonf', 'P(alguna alarma | sin drift)']:.0%}, BH {mt.loc['bh', 'P(alguna alarma | sin drift)']:.0%}.")
print(f"  Con 3 variables realmente derivadas, el crudo detecta {mt.loc['crudo', 'poder por variable con drift']:.0%} pero acusa {mt.loc['crudo', 'falsas alarmas por lote (con drift)']:.1f} variables sanas por lote (FDR {mt.loc['crudo', 'FDR observado']:.0%}); BH detecta {mt.loc['bh', 'poder por variable con drift']:.0%} con FDR {mt.loc['bh', 'FDR observado']:.0%}; Bonferroni {mt.loc['bonf', 'poder por variable con drift']:.0%}.")

Lectura (teoria: 1 - 0.95^50 = 92.3%):
  Sin ningun drift, el KS crudo da al menos una alarma en 92% de los lotes; Bonferroni 5%, BH 5%.
  Con 3 variables realmente derivadas, el crudo detecta 100% pero acusa 2.3 variables sanas por lote (FDR 44%); BH detecta 96% con FDR 6%; Bonferroni 91%.


## 9. Drift y desempeño del modelo: tres casos

Un modelo se entrena con $X_1,X_2,X_3\sim N(0,1)$. La verdad: $P(y=1\mid x)=\sigma\big(x_1+0{,}9\sin(2x_1)+0{,}8x_2\big)$ (**$X_3$ no influye**). Se entrena un modelo gradient boosting y se evalúa en tres tipos de cambio:

1. **Covariate shift relevante:** $X_1$ se desplaza $+\delta$ (el modelo extrapola donde hay pocos datos).
2. **Drift en una variable irrelevante:** $X_3$ se desplaza $+\delta$.
3. **Concept drift:** cambia $P(y\mid x)$ (el coeficiente de $x_2$ se invierte gradualmente) sin que **ninguna** distribución de entrada cambie.

In [16]:
def mundo(n, rg, d1=0.0, d3=0.0, g=0.0):
    X = rg.standard_normal((n, 3)); X[:, 0] += d1; X[:, 2] += d3
    z = X[:, 0] + 0.9 * np.sin(2 * X[:, 0]) + (0.8 - 1.6 * g) * X[:, 1]          # g=0: coeficiente +0.8; g=1: -0.8
    p = 1 / (1 + np.exp(-z)); return X, (rg.random(n) < p).astype(int), p

rg = np.random.default_rng(5); Xtr, ytr, _ = mundo(30_000, rg)
modelo = HistGradientBoostingClassifier(max_iter=250, learning_rate=0.08, random_state=0).fit(Xtr, ytr)
def evaluar(**kw):
    X, y, p = mundo(40_000, np.random.default_rng(99), **kw); pr = modelo.predict_proba(X)[:, 1]
    return {"PSI_x1": psi(Xtr[:, 0], X[:, 0]), "PSI_x3": psi(Xtr[:, 2], X[:, 2]), "AUC": roc_auc_score(y, pr), "logloss": log_loss(y, pr), "brier": brier_score_loss(y, pr), "logloss_oraculo": log_loss(y, p)}
filas = [{"caso": "base (sin cambio)", **evaluar()}]
for d in (0.5, 1.0, 1.5, 2.0): filas.append({"caso": f"covariate shift x1 +{d}", **evaluar(d1=d)})
for d in (1.0, 2.0): filas.append({"caso": f"x3 (irrelevante) +{d}", **evaluar(d3=d)})
for g in (0.25, 0.5, 1.0): filas.append({"caso": f"concept drift g={g}", **evaluar(g=g)})
imp = pd.DataFrame(filas).set_index("caso"); imp["exceso_logloss"] = imp["logloss"] - imp["logloss_oraculo"]; imp.round(3)

,PSI_x1,PSI_x3,AUC,logloss,brier,logloss_oraculo,exceso_logloss
caso,,,,,,,
base (sin cambio),0.0000,0.0000,0.8350,0.5030,0.1650,0.4980,0.0050
covariate shift x1 +0.5,0.2350,0.0000,0.8240,0.4960,0.1620,0.4920,0.0040
covariate shift x1 +1.0,0.9180,0.0000,0.7950,0.4770,0.1550,0.4710,0.0060
covariate shift x1 +1.5,1.9750,0.0000,0.7610,0.4550,0.1460,0.4440,0.0110
covariate shift x1 +2.0,3.3130,0.0000,0.7410,0.4240,0.1340,0.4030,0.0210
x3 (irrelevante) +1.0,0.0000,0.9420,0.8340,0.5040,0.1660,0.4980,0.0060
x3 (irrelevante) +2.0,0.0000,3.3200,0.8320,0.5060,0.1660,0.4980,0.0080
concept drift g=0.25,0.0000,0.0000,0.8150,0.5290,0.1740,0.5140,0.0150
concept drift g=0.5,0.0000,0.0000,0.7850,0.5710,0.1890,0.5180,0.0520


In [17]:
b = imp.loc["base (sin cambio)"]
print("Lectura:")
print(f"  Base: PSI ~0, AUC {b.AUC:.3f}, exceso de log-loss sobre el oraculo {b.exceso_logloss:.3f}.")
print(f"  Covariate shift en x1 (+2): PSI(x1) = {imp.loc['covariate shift x1 +2.0', 'PSI_x1']:.2f} y el exceso de log-loss pasa de {b.exceso_logloss:.3f} a {imp.loc['covariate shift x1 +2.0', 'exceso_logloss']:.3f}: el exceso se multiplica por {imp.loc['covariate shift x1 +2.0', 'exceso_logloss'] / b.exceso_logloss:.0f} (el modelo extrapola en una zona con pocos datos de entrenamiento), aunque en terminos absolutos sigue siendo pequeno.")
print(f"  Variable irrelevante (x3 +2): PSI(x3) = {imp.loc['x3 (irrelevante) +2.0', 'PSI_x3']:.2f} (alerta roja) pero el desempeno no cambia (exceso {imp.loc['x3 (irrelevante) +2.0', 'exceso_logloss']:.3f}): una alarma sin consecuencias.")
print(f"  Concept drift total (g=1): PSI(x1) = {imp.loc['concept drift g=1.0', 'PSI_x1']:.3f}, PSI(x3) = {imp.loc['concept drift g=1.0', 'PSI_x3']:.3f} (ninguna alarma) pero el AUC cae de {b.AUC:.2f} a {imp.loc['concept drift g=1.0', 'AUC']:.2f} y el log-loss sube de {b.logloss:.2f} a {imp.loc['concept drift g=1.0', 'logloss']:.2f}: dano SILENCIOSO.")
print("  El monitoreo de entradas no sustituye al de resultados: hace falta ademas vigilar el desempeno con etiquetas (con rezago) o proxies.")

Lectura:
  Base: PSI ~0, AUC 0.835, exceso de log-loss sobre el oraculo 0.005.
  Covariate shift en x1 (+2): PSI(x1) = 3.31 y el exceso de log-loss pasa de 0.005 a 0.021: el exceso se multiplica por 5 (el modelo extrapola en una zona con pocos datos de entrenamiento), aunque en terminos absolutos sigue siendo pequeno.
  Variable irrelevante (x3 +2): PSI(x3) = 3.32 (alerta roja) pero el desempeno no cambia (exceso 0.008): una alarma sin consecuencias.
  Concept drift total (g=1): PSI(x1) = 0.000, PSI(x3) = 0.000 (ninguna alarma) pero el AUC cae de 0.83 a 0.69 y el log-loss sube de 0.50 a 0.70: dano SILENCIOSO.
  El monitoreo de entradas no sustituye al de resultados: hace falta ademas vigilar el desempeno con etiquetas (con rezago) o proxies.


## 10. Pruebas automáticas

In [18]:
# 10.1 PSI: coincide con el valor poblacional; PSI(0)=0; creciente en delta; simetrico
assert np.abs(pp.PSI_muestral - pp.PSI_poblacional).max() < 0.01 and pp.PSI_poblacional.is_monotonic_increasing and abs(pp.loc[0.0, "PSI_poblacional"]) < 1e-12
a_, b_ = np.random.default_rng(1).standard_normal(5000), np.random.default_rng(2).standard_normal(5000) + 0.4
ed_ = np.quantile(a_, np.linspace(0, 1, 11)[1:-1])
assert abs(psi(a_, b_, edges=ed_) - psi(b_, a_, edges=ed_)) < 1e-12      # con las mismas cajas el PSI es simetrico
assert pp.loc[1.0, "PSI_poblacional"] < 1.0                                    # discretizar pierde informacion
# 10.2 ruido del PSI: media ~ (k-1)(2/n), y el umbral 0.1 falla con n chico
assert (np.abs(nula["PSI_medio"] / nula["aprox (k-1)(2/n)"] - 1) < 0.2).all() and nula.loc[100, "alerta_ambar(>0.1)"] > 0.4 and nula.loc[2000, "alerta_ambar(>0.1)"] < 0.01
# 10.3 KS: la tasa de rechazo crece con n con el mismo cambio minusculo; el PSI poblacional es minusculo
assert ks["rechazo_KS(p<.05)"].is_monotonic_increasing and ks.loc[500, "rechazo_KS(p<.05)"] < 0.15 and ks.loc[500_000, "rechazo_KS(p<.05)"] > 0.9 and ks.loc[500, "PSI_poblacional"] < 0.001
# 10.4 poder: la nula esta al 5%; la bimodal es invisible para media/varianza y visible para KS/PSI
assert poder.loc["bimodal (misma media/var)", "|t| medias"] < 0.12 and poder.loc["bimodal (misma media/var)", "razon var"] < 0.12 and poder.loc["bimodal (misma media/var)", "KS"] > 0.8
assert poder.loc["media +0.15", "|t| medias"] >= poder.loc["media +0.15", "PSI"] and poder.loc["desvio x1.3", "razon var"] > 0.9 and poder.loc["media +0.15", "razon var"] < 0.12
# 10.5 categoricas: el PSI crece al reducir eps
assert cat.iloc[:, 0].is_monotonic_increasing and cat.iloc[-1, 0] / cat.iloc[0, 0] > 2
# 10.6 multivariado: KS univariado ciego, clasificador detecta, sin cambio ~0.5
assert min(ks_vars) > 0.01 and auc_real > 0.8 and abs(auc_sin - 0.5) < 0.05 and auc_real > np.max(nulo_auc) + 0.2
# 10.7 multiples pruebas: crudo ~ 1-0.95^m; Bonferroni y BH controlan
assert abs(mt.loc["crudo", "P(alguna alarma | sin drift)"] - (1 - 0.95 ** m)) < 0.06 and mt.loc["bonf", "P(alguna alarma | sin drift)"] < 0.12 and mt.loc["bh", "P(alguna alarma | sin drift)"] < 0.12
assert mt.loc["crudo", "FDR observado"] > 0.4 and mt.loc["bh", "FDR observado"] < 0.12 and mt.loc["bh", "poder por variable con drift"] > mt.loc["bonf", "poder por variable con drift"] - 0.05
# 10.8 drift y desempeno: tres tipos de relacion
assert imp.loc["covariate shift x1 +2.0", "exceso_logloss"] > 3 * b.exceso_logloss and imp.loc["covariate shift x1 +2.0", "PSI_x1"] > 0.25
assert imp.loc["x3 (irrelevante) +2.0", "PSI_x3"] > 0.25 and abs(imp.loc["x3 (irrelevante) +2.0", "logloss"] - b.logloss) < 0.02
assert imp.loc["concept drift g=1.0", "PSI_x1"] < 0.02 and imp.loc["concept drift g=1.0", "PSI_x3"] < 0.02 and imp.loc["concept drift g=1.0", "AUC"] < b.AUC - 0.1
print("Todas las pruebas pasaron.")

Todas las pruebas pasaron.


## Conclusiones y límites

| Paso | Qué se encontró | Decisión |
|---|---|---|
| PSI | Valor poblacional exacto bajo normalidad; discretizar pierde información | Calibrar el umbral con la distribución nula según n |
| Tamaño de muestra | El PSI sin drift vale (k−1)(1/n+1/m): con n pequeño la alerta ámbar es ruido | No usar 0,1 y 0,25 como constantes universales |
| KS | Con n enorme rechaza cambios irrelevantes | Reportar magnitud (PSI/Wasserstein) además de significancia |
| Poder | Ninguna prueba cubre todos los cambios; la mezcla bimodal es invisible a media y varianza | Combinar KS/PSI con pruebas de momentos |
| Categóricas | El ε decide el PSI de categorías nuevas | Reportar categorías nuevas/perdidas aparte |
| Multivariado | Cambios de dependencia exigen un clasificador de dos muestras | Complementar las pruebas univariadas |
| Muchas variables | Con 50 variables el KS crudo alarma casi siempre | Controlar el FDR (BH) |
| Drift vs desempeño | Drift sin daño, daño sin drift y drift que daña | Vigilar entradas Y resultados |

**Límites que no deben ocultarse**

- Todas las distribuciones son sintéticas y los umbrales de la tabla de poder dependen del n, de los cambios elegidos y de la semilla.
- El clasificador de dos muestras usa una configuración fija de gradient boosting; con dimensión alta y poco n pierde poder.
- Los casos de impacto usan un solo modelo y un solo tipo de relación; en datos reales el daño depende de cómo use el negocio las predicciones.
- No se modela la dependencia temporal entre lotes consecutivos (autocorrelación).